# Task 10: Duplicate Record Check — Sample Superstore

**Internship:** Data Analytics Internship — Veda Technologies

This notebook checks the Sample Superstore dataset for exact duplicate rows, reviews possible duplicates using business-relevant columns, removes confirmed exact duplicate rows, and exports a duplicate report, cleaned dataset, and audit note.

**Important:** Repeated `Order ID` values alone are not necessarily duplicates because one order can contain multiple products. Exact full-row duplicates are the primary cleaning target. Review any key-based matches before removing them.

## 1. Import libraries and upload the dataset

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
from pathlib import Path
from IPython.display import display

In [ ]:
uploaded = files.upload()
file_name = next(iter(uploaded))
file_path = Path(file_name)

if file_path.suffix.lower() == ".csv":
    df = pd.read_csv(file_path)
elif file_path.suffix.lower() in [".xlsx", ".xls"]:
    df = pd.read_excel(file_path)
else:
    raise ValueError("Please upload a CSV or Excel file.")

df.columns = df.columns.astype(str).str.strip()
print("Loaded file:", file_name)
print("Dataset shape:", df.shape)
display(df.head())

## 2. Inspect the dataset

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nMissing values per column:")
display(df.isna().sum().to_frame("Missing Values"))

## 3. Detect exact duplicate rows

In [ ]:
duplicate_mask = df.duplicated(keep=False)
duplicate_rows = df.loc[duplicate_mask].copy()
exact_duplicate_rows_count = int(df.duplicated(keep="first").sum())

print("Rows belonging to exact-duplicate groups:", len(duplicate_rows))
print("Extra exact duplicate rows that would be removed:", exact_duplicate_rows_count)

display(duplicate_rows.head(50))

## 4. Review potential business-key duplicates

The code below looks for common Superstore identifiers. It uses Order ID + Product ID when both are present. If Product ID is unavailable, it checks Order ID + Product Name when both exist. These matches are only **potential** duplicates: inspect them before deciding whether they are data errors.

In [ ]:
normalized_columns = {col.lower().replace("_", " ").strip(): col for col in df.columns}

def find_column(candidates):
    for candidate in candidates:
        if candidate in normalized_columns:
            return normalized_columns[candidate]
    return None

order_id_col = find_column(["order id", "orderid"])
product_id_col = find_column(["product id", "productid"])
product_name_col = find_column(["product name", "productname"])
order_date_col = find_column(["order date", "orderdate"])

if order_id_col and product_id_col:
    key_columns = [order_id_col, product_id_col]
elif order_id_col and product_name_col:
    key_columns = [order_id_col, product_name_col]
else:
    key_columns = []

if key_columns:
    potential_key_mask = df.duplicated(subset=key_columns, keep=False)
    potential_key_duplicates = df.loc[potential_key_mask].sort_values(key_columns).copy()
    extra_key_duplicate_count = int(df.duplicated(subset=key_columns, keep="first").sum())
    print("Columns used for potential key check:", key_columns)
    print("Rows in repeated key groups:", len(potential_key_duplicates))
    print("Extra rows beyond the first within repeated key groups:", extra_key_duplicate_count)
    display(potential_key_duplicates.head(50))
else:
    potential_key_duplicates = pd.DataFrame()
    extra_key_duplicate_count = 0
    print("Could not identify a suitable order/product key combination. Exact full-row duplicate checking is still complete.")

## 5. Create a duplicate report

In [ ]:
if not duplicate_rows.empty:
    duplicate_report = duplicate_rows.copy()
    duplicate_report.insert(0, "Source Row Number", duplicate_rows.index + 2)
else:
    duplicate_report = pd.DataFrame(columns=["Source Row Number"] + df.columns.tolist())

print("Duplicate report shape:", duplicate_report.shape)
display(duplicate_report.head(20))

## 6. Remove exact duplicate rows

In [ ]:
cleaned_df = df.drop_duplicates(keep="first").copy()
removed_rows = len(df) - len(cleaned_df)
remaining_exact_duplicates = int(cleaned_df.duplicated().sum())

print("Original row count:", len(df))
print("Cleaned row count:", len(cleaned_df))
print("Exact duplicate rows removed:", removed_rows)
print("Remaining exact duplicates:", remaining_exact_duplicates)

display(cleaned_df.head())

## 7. Create an audit note

In [ ]:
audit_note = pd.DataFrame([
    {"Metric": "Source file", "Value": file_name},
    {"Metric": "Original row count", "Value": len(df)},
    {"Metric": "Original column count", "Value": len(df.columns)},
    {"Metric": "Rows belonging to exact-duplicate groups", "Value": len(duplicate_rows)},
    {"Metric": "Extra exact duplicate rows removed", "Value": removed_rows},
    {"Metric": "Cleaned row count", "Value": len(cleaned_df)},
    {"Metric": "Remaining exact duplicates", "Value": remaining_exact_duplicates},
    {"Metric": "Business-key columns reviewed", "Value": ", ".join(key_columns) if key_columns else "Not identified"},
    {"Metric": "Extra repeated-key rows (review only)", "Value": extra_key_duplicate_count},
    {"Metric": "Cleaning rule", "Value": "Removed exact full-row duplicates; retained first occurrence."},
    {"Metric": "Key-check caution", "Value": "Repeated order/product keys are potential matches and were not automatically removed."}
])

display(audit_note)

## 8. Export deliverables

In [ ]:
duplicate_report_file = "superstore_duplicate_report.csv"
cleaned_file = "superstore_cleaned.csv"
audit_file = "duplicate_check_audit_note.csv"

duplicate_report.to_csv(duplicate_report_file, index=False)
cleaned_df.to_csv(cleaned_file, index=False)
audit_note.to_csv(audit_file, index=False)

print("Created files:")
print(duplicate_report_file)
print(cleaned_file)
print(audit_file)

## 9. Download the output files

In [ ]:
files.download(duplicate_report_file)
files.download(cleaned_file)
files.download(audit_file)

## Conclusion
This notebook checks exact duplicate rows, records the results, removes exact duplicates only, and exports a cleaned dataset with an audit note. Review potential order/product-key matches separately before making any additional deletions.